# Exploring a NASA CRM CGNS grid with pyramids-mesh

**CGNS** — the *CFD General Notation System* — is the standard interchange format of computational fluid
dynamics. Grid generators and flow solvers across the CFD world write it, storing the mesh (and any flow
solution) as a SIDS node tree inside an **HDF5** container. Passing `driver="cgns"` sends pyramids-mesh down
the HDF5 path, reading the grid through the vendored GDAL multidimensional driver and handing back a
`MeshDataset` — the same pyramids object every other driver produces.

The file we open is a genuine research grid: the **NASA DPW4 Common Research Model (CRM) wing-body**, the
transonic test geometry from the 4th AIAA Drag Prediction Workshop. Its CGNS zone is a mixed-element volume
mesh wrapped in boundary patches — the **surface** that pyramids-mesh reads is a mix of **quadrilateral
(`QUAD_4`) and triangular (`TRI_3`) patches** covering the wing, the fuselage, the symmetry plane and the
farfield box.

This notebook reads the coarsest grid of the family, confirms the mesh comes back **compact** (a historical
orphan-node defect, issue #97, is now fixed), tours its mixed-element topology, and then shows that the whole
pyramids GIS layer applies to it for free.

> Source: NASA DPW4 CRM wing-body grids, Zenodo record 8280935 (`cgnsGrids.zip`). The coordinates are the
> solver's own model units with no map projection, so `crs`/`epsg` are `None` throughout.

## 1. Imports

`MeshDataset` is the object the `cgns` driver hands back. Importing from `pyramids` also puts its vendored
GDAL/PROJ data in place, so nothing here is order-sensitive in a notebook — just import the names you use.
`add_tiles` would add a web basemap *when a mesh is georeferenced*; a CGNS grid never is, so we only ever
call it behind an `if ds.epsg` guard.

In [ ]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pyramids.netcdf.ugrid import MeshDataset
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the grid

This notebook lives in `docs/examples/real-data/`, so the shared `examples/data/real-data/` corpus is three
levels up. The CGNS grids live under `cgns/`, with filenames encoding the geometry, the element mix and the
refinement level. Paths inside a notebook are written **relative to the notebook itself**, and pyramids'
`read_file` takes a `pathlib.Path` directly — we pass the `Path`, never `str(path)`.

We read only the coarsest grid, `g0-coarse`. The finer `g1`/`g2`/`g3` grids (450 MB … 1.2 GB) sit in the
same folder — they are refinements of the *identical* geometry and everything below applies to them
unchanged, so there is no need to open them here.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
grid = DATA / "cgns" / "cgns_dpw4crm_unstructured_mixed_g0-coarse_2.07mn_orphan98.cgns"
grid, grid.exists()

## 3. Read the grid

`MeshDataset.read_file(path, driver="cgns")` dispatches the file to the `cgns` driver, which walks the HDF5
SIDS tree, assembles the surface patches into a face table, and returns a `MeshDataset`. We time the read: the
file is 263 MB on disk, yet only its surface is materialised, so the read is fast.

You will see warnings that some faces were re-wound to counter-clockwise and that some degenerate faces were
left unchanged — that is pyramids normalising the winding order, informational rather than an error.

In [ ]:
t0 = time.perf_counter()
ds = MeshDataset.read_file(grid, driver="cgns")
read_seconds = time.perf_counter() - t0
print(f"read in {read_seconds:.2f} s")
ds

A 263 MB CGNS file reads into a 2-D `MeshDataset` in about a second: **50 316 nodes, 58 480 faces**, with a
single `node_z` data variable — the nodal z-coordinate of the 3-D wing-body surface lifted into a field.

## 4. Choose the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Immediately after that
first read we call cleopatra's own chooser once: it detects the notebook and selects the inline backend (and
falls back to `Agg` in a plain script or in CI), so every figure below embeds natively with no manual
rasterising.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 5. Explore the dataset

A `MeshDataset` exposes summary properties describing the whole mesh at a glance: `n_node` / `n_face` (size),
`crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (spatial extent), and
`data_variable_names` (any attached fields).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(b, 2) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

`crs`/`epsg` are `None` — a CGNS grid lives in the solver's own Cartesian coordinates with no datum, so
reprojection and basemap tiles are off the table. The `bounds` span roughly **x ∈ [-15, 16]**, **y ∈ [0, 15]**
in model metres: the aircraft sits near the origin and the farfield box patches reach out to the edges. The
one data variable, `node_z`, is the surface elevation we will map below.

## 6. Is the mesh compact? (the issue-#97 orphan-node fix)

These DPW4 grids were the data behind issue #97: the CGNS zone is a mixed-element **volume** mesh, but the
driver reads only the **surface** patches. Earlier the reader kept *every* coordinate node — all ~2.07 million
of them — while the faces referenced only the surface skin, leaving ~98% of the nodes **orphaned** (used by no
face). That inflated `n_node` and carried a mostly-unused `node_z` array.

The honest way to check this is to compare `n_node` against the set of node indices the faces actually
reference. `face_node_connectivity` is an `(n_face, k)` table padded with `-1`, so we take the unique
non-negative indices and count them.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
referenced = np.unique(conn[conn >= 0])
pd.Series({
    "n_node": ds.n_node,
    "unique_referenced_nodes": referenced.size,
    "orphan_nodes": ds.n_node - referenced.size,
    "every_node_used": bool(ds.n_node == referenced.size),
})

**Every one of the 50 316 nodes appears in the face connectivity** — zero orphans. The orphan-node issue is
now fixed: rather than the ~2.07 million nodes the historical defect reported (≈98% unused), the reader hands
back the compact set of exactly the nodes the surface uses, so `node_z` and every downstream computation only
ever touch real surface points.

## 7. A mixed-element surface

The CRM surface is not one element type — the patches are a mix of quadrilaterals and triangles. The
connectivity table has a fixed width `k = 4` (the widest face), and each **triangle** fills three slots and
pads the fourth with `-1`. Counting the valid (non-`-1`) entries per row tells us the element mix directly.

In [ ]:
widths = (conn >= 0).sum(axis=1)
mix = (
    pd.Series(widths)
    .value_counts()
    .rename_axis("nodes_per_face")
    .rename({3: "3 (TRI_3)", 4: "4 (QUAD_4)"})
    .to_frame("n_faces")
)
print("connectivity shape:", conn.shape)
mix

The table is `(58480, 4)`: about **42 000 quadrilaterals** (all four columns filled) and **16 000 triangles**
(three columns plus a `-1` pad). A single `MeshDataset` holds both element types side by side — the reader
does not split them into separate meshes.

## 8. The wireframe

The simplest visualisation needs no data — just draw the cell edges. `plot_outline()` renders the mesh
skeleton through cleopatra. At 58 480 faces this is a dense drawing, but still quick, and it is the fastest
confirmation that the surface parsed correctly. The tile call is guarded by `if ds.epsg`, so it is skipped for
this un-georeferenced grid.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)
_g.ax.set_title("NASA CRM CGNS surface — wireframe")
_g.ax.set_aspect("equal")

The wireframe is the surface as the solver sees it: the dense, refined cluster near the origin is the
wing-body itself, and the sparse large cells reaching to the edges are the farfield boundary patches. The
half-model stops at the `y = 0` symmetry plane — only one wing is meshed.

## 9. A data map: surface elevation

Unlike a bare mesh-generator file, this grid already carries a field: `node_z`, the z-coordinate of each
surface node. Colouring the mesh by it draws the surface elevation of the wing-body — the generic pyramids
data-map that any per-node field (pressure, temperature, …) would use. `summary()` is the numeric companion:
a per-variable `DataFrame` of count / min / max / mean / std.

In [ ]:
_g = ds.plot("node_z", cmap="terrain", colorbar=True, title="CRM surface — node_z (elevation)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)
_g.ax.set_aspect("equal")

In [ ]:
ds.summary()

The colour map picks out the raised wing and fuselage against the flat symmetry/farfield patches, and the
`summary()` row confirms it quantitatively: `node_z` spans roughly ±15 m about a mean near zero across all
50 316 surface nodes — every value a genuine surface point, none left over from orphaned interior nodes.

## 10. GIS operations, for free

Because the mesh is a pyramids object, the pyramids GIS layer applies directly — none of these methods live in
`pyramids-mesh`, which only reads the file. We tour four.

### Faces as a `GeoDataFrame`

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` — the standard vector
structure for export to shapefile/GeoPackage or for spatial joins. Both quads and triangles become ordinary
shapely polygons.

In [ ]:
gdf = ds.to_geodataframe()
print(type(gdf).__name__, gdf.shape)
gdf.head()

One polygon per face, 58 480 rows. The CRS is unset — inherited from the mesh's `None` — so you would assign
one before exporting if the grid were georeferenced (this one is not).

### The outer boundary

`bounding_polygon()` dissolves all faces into the single enclosing polygon — here the outline of the farfield
box the surface patches span.

In [ ]:
poly = ds.bounding_polygon()
ax = gpd.GeoSeries([poly]).plot(facecolor="none", edgecolor="red", linewidth=2)
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
ax.set_aspect("equal")
ax.set_title("bounding polygon")
pd.Series({"geom_type": poly.geom_type, "area": round(poly.area, 1), "bounds": tuple(round(b, 1) for b in ds.bounds)})

A single `Polygon` enclosing the whole domain — an ordinary shapely geometry, the gateway to the
shapely/geopandas ecosystem for clipping and spatial joins.

### The face-adjacency graph

`connected_components()` labels which faces are reachable from which through shared edges, and
`sparse_adjacency()` returns the face-to-face adjacency as a SciPy sparse matrix. A watertight surface is one
single component, which is the quickest topological health check on a grid.

In [ ]:
adj = ds.sparse_adjacency()
comp = ds.connected_components()
pd.Series({
    "adjacency_shape": adj.shape,
    "adjacency_nnz": adj.nnz,
    "connected_components": int(np.unique(comp).size),
})

**One** connected component: the wing-body surface and its farfield patches stitch into a single edge-connected
shell, with every face wired to its neighbours — exactly what a clean, compact surface should report.

### Cropping to the aircraft

`crop(bbox=...)` returns a new `MeshDataset` with only the faces inside the box. The farfield patches sprawl
out to ±15, while the wing-body sits in a tight region near the origin, so cropping to a small central box
cleanly separates the aircraft from the outer boundary.

In [ ]:
aircraft = ds.crop(bbox=(-2.0, 0.0, 4.0, 6.0))
print("faces before / after:", ds.n_face, "->", aircraft.n_face)
_g = aircraft.plot_outline()
_g.ax.set_title("cropped to the wing-body")
_g.ax.set_aspect("equal")

The crop drops the ~37 000 farfield-box faces and keeps the ~22 000 that make up the aircraft, and the
wireframe now shows the wing and fuselage on their own — a region extracted with one call, ready for a
closer look or a separate export.

## 11. The finer grids

The same folder holds `g1-medium`, `g2-fine` and `g3-extrafine` — the identical CRM geometry at 3.8, 6.5 and
10.5 million volume nodes. They read exactly as `g0-coarse` does, returning progressively denser surface
meshes (≈98k, 149k and 222k faces), and every operation in this notebook applies to them unchanged. We leave
them on disk here: the coarse grid already tells the whole story, and the finer files are 450 MB–1.2 GB each.

> **Note — xarray interop.** pyramids-core exposes a `to_xarray()` bridge to the xarray ecosystem, but these
> notebooks deliberately stay within pyramids-core's own GDAL-based UGRID I/O and do not import `xarray` (nor
> `h5py`, `netCDF4`, or `scipy`) directly.

## Takeaway

- `driver="cgns"` reads the CFD-standard **HDF5** CGNS container — here a 263 MB NASA DPW4 CRM wing-body grid —
  into a `MeshDataset` in about a second, materialising only the surface.
- The surface is **mixed-element**: ~42 000 `QUAD_4` quadrilaterals and ~16 000 `TRI_3` triangles share one
  `(n_face, 4)` connectivity table, with triangles padded by `-1`.
- The grid comes back **compact**: all 50 316 nodes are referenced by faces, so the historical issue-#97
  orphan-node defect (~98% unused nodes, `n_node` inflated to ~2.07 million) is fixed.
- The coordinates are solver-local (`crs`/`epsg` are `None`), so reprojection and basemap tiles are skipped —
  yet the full pyramids GIS layer (`to_geodataframe`, `bounding_polygon`, `connected_components`, `crop`)
  applies to the mesh for free.
- Winding-normalisation warnings on read are informational, not failures.